# WeatherForYnov — Construction du dataset LSTM enrichi

**Hackathon YNOV — Sujet 2 : Prévisions météorologiques**

Ce notebook reconstruit depuis les fichiers **MENSQ bruts** un dataset adapté au LSTM multi-sorties.

## Variables cibles (outputs du LSTM)
| Variable MENSQ | Description officielle Météo-France | Alias dataset |
|---|---|---|
| **TMM** | Moyenne mensuelle des températures moyennes quotidiennes | `tmm` |
| **TX** | Moyenne mensuelle des températures maximales quotidiennes | `tx` |
| **TN** | Moyenne mensuelle des températures minimales quotidiennes | `tn` |

## Features enrichies (inputs du LSTM)
- `TXAB` — Max absolu mensuel des TX quotidiennes (record de chaleur du mois)
- `TNAB` — Min absolu mensuel des TN quotidiennes (record de froid du mois)
- `TMMAX / TMMIN` — Max/min mensuel de la température moyenne quotidienne
- `TAMPLIM` — Amplitude thermique mensuelle moyenne
- `NBJTX25`, `NBJTX30` — Jours de chaleur (≥ 25°C, ≥ 30°C)
- `NBJGELEE` — Jours de gelée
- `NBJTN5` — Jours de grand froid (TN ≤ -5°C)
- `RR` — Précipitations cumulées
- `UMM` — Humidité relative moyenne
- `FFM` — Vent moyen
- `INST` — Insolation mensuelle
- `PMERM` — Pression atmosphérique moyenne

## Paramètres
- **Période** : 2001–2025
- **Qualité** : codes Q = 1 uniquement
- **Export** : `src/data/processed/climat_lstm_targets.csv`

---
## 1. Configuration et imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR      = ROOT / "src" / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GEO_PATH    = DATA_DIR / "referentiel_geo_departements.csv"
OUTPUT_CSV  = PROCESSED_DIR / "climat_lstm_targets.csv"

YEAR_MIN = 2001
YEAR_MAX = 2025

print(f"Racine projet : {ROOT}")
print(f"Dossier données brutes : {DATA_DIR}")
print(f"Export : {OUTPUT_CSV}")
print(f"Période : {YEAR_MIN}–{YEAR_MAX}")

---
## 2. Définition du schéma de colonnes

In [ ]:
# Colonnes identifiantes
ID_COLS = ["NUM_POSTE", "NOM_USUEL", "LAT", "LON", "ALTI", "AAAAMM"]

# Variables cibles (3 outputs du LSTM)
TARGET_COLS = [
    "TMM",    # Temp. moyenne mensuelle des moyennes quotidiennes
    "TX",     # Temp. moyenne mensuelle des maximales quotidiennes
    "TN",     # Temp. moyenne mensuelle des minimales quotidiennes
]

# Codes qualité associés aux cibles (pour filtrage Q=1)
TARGET_Q_COLS = ["QTMM", "QTX", "QTN"]

# Features enrichies
FEATURE_COLS = [
    "TXAB",       # Max absolu mensuel des TX quotidiennes (record de chaleur)
    "TNAB",       # Min absolu mensuel des TN quotidiennes (record de froid)
    "TMMAX",      # Max mensuel de (TN+TX)/2 quotidien
    "TMMIN",      # Min mensuel de (TN+TX)/2 quotidien
    "TAMPLIM",    # Amplitude thermique mensuelle moyenne
    "NBJTX25",    # Nb jours TX ≥ 25°C
    "NBJTX30",    # Nb jours TX ≥ 30°C
    "NBJTX35",    # Nb jours TX ≥ 35°C (canicule)
    "NBJGELEE",   # Nb jours de gelée (TN ≤ 0°C)
    "NBJTN5",     # Nb jours TN ≤ -5°C (grand froid)
    "NBJTN10",    # Nb jours TN ≤ -10°C (froid extrême)
    "RR",         # Précipitations cumulées (mm)
    "RRAB",       # Précipitations max en 24h
    "NBJRR1",     # Nb jours avec pluie ≥ 1mm
    "UMM",        # Humidité relative moyenne
    "FFM",        # Vent moyen mensuel
    "FXIAB",      # Rafale max mensuelle
    "INST",       # Insolation mensuelle (minutes)
    "PMERM",      # Pression atmosphérique moyenne
    "ETP",        # Évapotranspiration
]

ALL_COLS = ID_COLS + TARGET_COLS + TARGET_Q_COLS + FEATURE_COLS

print(f"Colonnes identifiantes : {len(ID_COLS)}")
print(f"Variables cibles       : {TARGET_COLS}")
print(f"Features enrichies     : {len(FEATURE_COLS)}")
print(f"Total colonnes à lire  : {len(ALL_COLS)}")

---
## 3. Chargement de tous les fichiers MENSQ bruts

In [ ]:
def load_mensq_file(filepath: Path, needed_cols: list) -> pd.DataFrame:
    """Charge un fichier MENSQ brut (séparateur ;) et retourne uniquement les colonnes disponibles."""
    try:
        df = pd.read_csv(filepath, sep=";", low_memory=False)
        # Ne garder que les colonnes qui existent dans ce fichier
        available = [c for c in needed_cols if c in df.columns]
        missing   = [c for c in needed_cols if c not in df.columns]
        df = df[available].copy()
        # Ajouter les colonnes manquantes en NaN
        for col in missing:
            df[col] = np.nan
        return df[needed_cols]
    except Exception as e:
        print(f"  ⚠ Erreur lecture {filepath.name} : {e}")
        return pd.DataFrame(columns=needed_cols)


# Lister tous les fichiers MENSQ (previous + latest)
mensq_files = sorted(DATA_DIR.glob("MENSQ_*.csv"))
print(f"Fichiers MENSQ trouvés : {len(mensq_files)}")

# Chargement en batch
chunks = []
for i, fp in enumerate(mensq_files):
    df_chunk = load_mensq_file(fp, ALL_COLS)
    if len(df_chunk) > 0:
        chunks.append(df_chunk)
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(mensq_files)} fichiers chargés...")

df_raw = pd.concat(chunks, ignore_index=True)
print(f"\nDimensions brutes concaténées : {df_raw.shape}")
print(f"Mémoire utilisée             : {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")

---
## 4. Décodage des dates et filtrage de la période

In [ ]:
# AAAAMM est au format 202501 (YYYYMM)
df_raw["AAAAMM"] = df_raw["AAAAMM"].astype(str).str.strip()
df_raw["annee"]  = df_raw["AAAAMM"].str[:4].astype(int)
df_raw["mois"]   = df_raw["AAAAMM"].str[4:6].astype(int)
df_raw["date"]   = pd.to_datetime(
    df_raw["annee"].astype(str) + "-" + df_raw["mois"].astype(str).str.zfill(2) + "-01"
)

# Filtrage 2001–2025
mask_period = (df_raw["annee"] >= YEAR_MIN) & (df_raw["annee"] <= YEAR_MAX)
df = df_raw[mask_period].copy()

print(f"Avant filtrage période : {len(df_raw):,} lignes")
print(f"Après filtrage {YEAR_MIN}–{YEAR_MAX} : {len(df):,} lignes")
print(f"Années présentes : {sorted(df['annee'].unique())}")

---
## 5. Conversion des colonnes numériques (valeurs ÷ 10)

In [ ]:
# Les températures et précipitations sont en 1/10 d'unité dans les fichiers MENSQ
# Voir descriptif PDF : précision 1/10 → diviser par 10 pour obtenir °C et mm

TEMP_COLS = [
    "TMM", "TX", "TN",
    "TXAB", "TNAB", "TMMAX", "TMMIN", "TAMPLIM"
]
PRECIP_COLS = ["RR", "RRAB"]
PRESS_COLS  = ["PMERM"]
WIND_COLS   = ["FFM", "FXIAB"]
ETP_COLS    = ["ETP"]

DIVIDE_BY_10 = TEMP_COLS + PRECIP_COLS + PRESS_COLS + WIND_COLS + ETP_COLS

for col in DIVIDE_BY_10:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce") / 10.0

# Colonnes entières (pas de division)
INT_COLS = [
    "NBJTX25", "NBJTX30", "NBJTX35",
    "NBJGELEE", "NBJTN5", "NBJTN10",
    "NBJRR1", "INST", "UMM"
]
for col in INT_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Codes qualité
for col in TARGET_Q_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Conversion des unités effectuée.")
print("\nAperçu températures (°C) :")
display(df[["date", "NUM_POSTE", "TMM", "TX", "TN", "TXAB", "TNAB"]].head(5))

---
## 6. Filtrage qualité Météo-France (Q = 1)

In [ ]:
n_before = len(df)

# Garder uniquement les lignes où les 3 cibles ont Q=1
# Q=1 : valeur certifiée | Q=0 : manquante | Q=9 : douteuse
q_mask = (
    (df["QTMM"].isin([1.0])) &
    (df["QTX"].isin([1.0]))  &
    (df["QTN"].isin([1.0]))
)

df = df[q_mask].copy()
n_after = len(df)

print(f"Avant filtrage qualité : {n_before:,} lignes")
print(f"Après filtrage Q=1 sur TMM/TX/TN : {n_after:,} lignes")
print(f"Lignes retirées : {n_before - n_after:,} ({(n_before - n_after) / n_before * 100:.1f}%)")

# Supprimer les colonnes qualité (plus utiles)
df = df.drop(columns=TARGET_Q_COLS)

---
## 7. Déduplication et cohérence des séries

In [ ]:
# Supprimer les doublons station × mois (previous + latest se chevauchent)
n_before = len(df)
df = df.sort_values(["NUM_POSTE", "date", "AAAAMM"])
df = df.drop_duplicates(subset=["NUM_POSTE", "AAAAMM"], keep="last")  # garder la version la plus récente
df = df.sort_values(["NUM_POSTE", "date"]).reset_index(drop=True)

print(f"Avant déduplication : {n_before:,} lignes")
print(f"Après déduplication : {len(df):,} lignes")
print(f"Doublons supprimés  : {n_before - len(df):,}")
print(f"Stations uniques    : {df['NUM_POSTE'].nunique():,}")

---
## 8. Filtrage des stations avec séries complètes

In [ ]:
# Nombre de mois par station
station_counts = df.groupby("NUM_POSTE")["date"].count().rename("n_mois")

# Pour le LSTM : fenêtre 36 mois + horizon 12 mois = 48 mois minimum
# On filtre les stations avec au moins 10 ans de données (120 mois) pour robustesse
MIN_MONTHS = 120
valid_stations = station_counts[station_counts >= MIN_MONTHS].index

df = df[df["NUM_POSTE"].isin(valid_stations)].copy()

print(f"Distribution du nombre de mois par station :")
print(station_counts.describe().round(0))
print(f"\nStations avec >= {MIN_MONTHS} mois : {len(valid_stations):,}")
print(f"Lignes conservées               : {len(df):,}")

---
## 9. Fusion du référentiel géographique

In [ ]:
if GEO_PATH.exists():
    df_geo = pd.read_csv(GEO_PATH, dtype=str)
    print(f"Référentiel géo chargé : {df_geo.shape}")
    print(f"Colonnes : {list(df_geo.columns)}")

    # Extraire le code département depuis NUM_POSTE (2 premiers caractères)
    df["code_departement"] = df["NUM_POSTE"].astype(str).str.zfill(8).str[:2]

    # Identifier la colonne département dans le référentiel
    dep_col = next(
        (c for c in df_geo.columns if "dep" in c.lower() and "code" in c.lower()),
        df_geo.columns[0]
    )
    print(f"\nColonne département utilisée pour le merge : '{dep_col}'")

    df_geo = df_geo.rename(columns={dep_col: "code_departement"})
    df = df.merge(df_geo, on="code_departement", how="left")
    print(f"Après fusion géo : {df.shape}")
else:
    print(f"⚠ Référentiel géo non trouvé : {GEO_PATH}")
    print("  Poursuite sans enrichissement géographique.")
    df["code_departement"] = df["NUM_POSTE"].astype(str).str.zfill(8).str[:2]

display(df.head(3))

---
## 10. Renommage final et nettoyage du schéma

In [ ]:
RENAME_MAP = {
    "NOM_USUEL"  : "nom_station",
    "LAT"        : "lat",
    "LON"        : "lon",
    "ALTI"       : "alti",
    # Cibles
    "TMM"        : "temp_moy_quotidienne",   # Moyenne des moyennes journalières
    "TX"         : "temp_max_quotidienne",   # Moyenne des maximales journalières
    "TN"         : "temp_min_quotidienne",   # Moyenne des minimales journalières
    # Features enrichies
    "TXAB"       : "temp_max_absolu",        # Record de chaleur du mois
    "TNAB"       : "temp_min_absolu",        # Record de froid du mois
    "TMMAX"      : "temp_moy_max",           # Jour le plus chaud (moyenne)
    "TMMIN"      : "temp_moy_min",           # Jour le moins chaud (moyenne)
    "TAMPLIM"    : "amplitude_thermique",    # Amplitude TX-TN moyenne
    "NBJTX25"    : "nb_jours_tx25",
    "NBJTX30"    : "nb_jours_tx30",
    "NBJTX35"    : "nb_jours_tx35",
    "NBJGELEE"   : "nb_jours_gelee",
    "NBJTN5"     : "nb_jours_tn5",
    "NBJTN10"    : "nb_jours_tn10",
    "RR"         : "precipitations_mm",
    "RRAB"       : "precip_max_24h",
    "NBJRR1"     : "nb_jours_pluie",
    "UMM"        : "humidite",
    "FFM"        : "vent_moyen",
    "FXIAB"      : "rafale_max",
    "INST"       : "insolation_min",
    "PMERM"      : "pression_mer",
    "ETP"        : "evapotranspiration",
    "AAAAMM"     : "aaaamm",
}

df = df.rename(columns=RENAME_MAP)

# Ordre final des colonnes
ID_FINAL = ["NUM_POSTE", "nom_station", "lat", "lon", "alti",
            "code_departement", "annee", "mois", "date"]
TARGET_FINAL = ["temp_moy_quotidienne", "temp_max_quotidienne", "temp_min_quotidienne"]
FEATURE_FINAL = [
    "temp_max_absolu", "temp_min_absolu", "temp_moy_max", "temp_moy_min",
    "amplitude_thermique",
    "nb_jours_tx25", "nb_jours_tx30", "nb_jours_tx35",
    "nb_jours_gelee", "nb_jours_tn5", "nb_jours_tn10",
    "precipitations_mm", "precip_max_24h", "nb_jours_pluie",
    "humidite", "vent_moyen", "rafale_max",
    "insolation_min", "pression_mer", "evapotranspiration",
]

# Ajouter les colonnes géo si présentes
geo_cols = [c for c in df.columns if c not in ID_FINAL + TARGET_FINAL + FEATURE_FINAL + ["aaaamm"]]
geo_keep = [c for c in geo_cols if any(k in c.lower() for k in ["dep", "region", "nom_dep", "nom_reg"])]

FINAL_COLS = ID_FINAL + geo_keep + TARGET_FINAL + FEATURE_FINAL
FINAL_COLS = [c for c in FINAL_COLS if c in df.columns]  # garder celles présentes

df = df[FINAL_COLS].copy()
print(f"Schéma final : {df.shape}")
print(f"Colonnes     : {list(df.columns)}")
display(df.head(3))

---
## 11. Analyse de complétude du dataset final

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

# Taux de valeurs manquantes
missing_pct = df[TARGET_FINAL + FEATURE_FINAL].isnull().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["tomato" if p > 50 else "steelblue" if p > 20 else "seagreen" for p in missing_pct]
missing_pct.plot(kind="bar", ax=ax, color=colors, edgecolor="white")
ax.axhline(50, color="red", linestyle="--", alpha=0.6, label="50% manquants")
ax.axhline(20, color="orange", linestyle="--", alpha=0.6, label="20% manquants")
ax.set_title("Taux de valeurs manquantes par variable (%)", fontsize=12, fontweight="bold")
ax.set_ylabel("%")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print("\nTaux de manquants :")
display(missing_pct.to_frame("% NaN").round(1))

In [ ]:
print("=" * 60)
print("   STATISTIQUES DES 3 VARIABLES CIBLES")
print("=" * 60)
display(df[TARGET_FINAL].describe().round(2))

print("\nDistribution temporelle :")
print(df.groupby("annee")["NUM_POSTE"].nunique().rename("nb_stations").to_string())

---
## 12. Export du dataset enrichi

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)

file_size_mb = OUTPUT_CSV.stat().st_size / 1e6

print("=" * 60)
print("   DATASET LSTM ENRICHI — EXPORT RÉUSSI")
print("=" * 60)
print(f"  Fichier    : {OUTPUT_CSV}")
print(f"  Taille     : {file_size_mb:.1f} MB")
print(f"  Lignes     : {len(df):,}")
print(f"  Colonnes   : {df.shape[1]}")
print(f"  Stations   : {df['NUM_POSTE'].nunique():,}")
print(f"  Période    : {df['annee'].min()} – {df['annee'].max()}")
print("")
print("  Variables cibles (3 outputs LSTM) :")
for t in TARGET_FINAL:
    n_ok = df[t].notna().sum()
    pct  = n_ok / len(df) * 100
    print(f"    {t:<30} {n_ok:>8,} valides ({pct:.1f}%)")
print("=" * 60)

---
## 13. Prochaine étape

Le dataset `climat_lstm_targets.csv` est prêt. Il contient :
- **3 variables cibles** : `temp_moy_quotidienne`, `temp_max_quotidienne`, `temp_min_quotidienne`
- **20 features d'entrée** enrichies depuis les MENSQ bruts
- **Qualité certifiée** Q=1 sur les 3 cibles
- **Période 2001–2025**, séries ≥ 10 ans par station

→ Le notebook `04_lstm_multi_output.ipynb` utilisera ce fichier pour entraîner le LSTM 2 couches avec fenêtre 36 mois et 3 sorties simultanées.